**1.Load and Clean Data**

Detect the encoding of the CSV files

In [1]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

csv_file1_path = 'Movies.csv'
csv_file2_path = 'Ratings.csv'
csv_file3_path = 'Users.csv'

encoding1 = detect_encoding(csv_file1_path)
encoding2 = detect_encoding(csv_file2_path)
encoding3 = detect_encoding(csv_file3_path)

print(f'Encoding of {csv_file1_path}: {encoding1}')
print(f'Encoding of {csv_file2_path}: {encoding2}')
print(f'Encoding of {csv_file3_path}: {encoding3}')


Encoding of Movies.csv: ISO-8859-1
Encoding of Ratings.csv: ascii
Encoding of Users.csv: ascii


Read the CSV files with the detected encoding

In [2]:
import pandas as pd

# Read CSV files with detected encoding
df1 = pd.read_csv(csv_file1_path, encoding=encoding1)
df2 = pd.read_csv(csv_file2_path, encoding=encoding2)
df3 = pd.read_csv(csv_file3_path, encoding=encoding3)

# Display initial data
print(df1.head())
print(df2.head())
print(df3.head())


   MovieID                               Title                      Category  \
0        1                    Toy Story (1995)   Animation|Children's|Comedy   
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy   
2        3             Grumpier Old Men (1995)                Comedy|Romance   
3        4            Waiting to Exhale (1995)                  Comedy|Drama   
4        5  Father of the Bride Part II (1995)                        Comedy   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
   UserID  MovieID  Rating
0       1     1193       5
1       1      661       3
2       1      914       3
3       1     3408       4
4       1     2355       5
   UserID Gender  Age  Occupation
0       1      F    1          10
1       2      M   56          16
2       3      M   25          15
3       4      M   45           7
4       5      M   25          

Clean data 

In [3]:
# Clean data (example: drop duplicates and handle missing values)
df1.drop_duplicates(inplace=True)
df2.drop_duplicates(inplace=True)
df3.drop_duplicates(inplace=True)

df1.fillna('', inplace=True)
df2.fillna(0, inplace=True)
df3.fillna('', inplace=True)

print(df1.info())
print(df2.info())
print(df3.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   MovieID     3883 non-null   int64 
 1   Title       3883 non-null   object
 2   Category    3883 non-null   object
 3   Unnamed: 3  3883 non-null   object
 4   Unnamed: 4  3883 non-null   object
dtypes: int64(1), object(4)
memory usage: 151.8+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 3 columns):
 #   Column   Non-Null Count    Dtype
---  ------   --------------    -----
 0   UserID   1000209 non-null  int64
 1   MovieID  1000209 non-null  int64
 2   Rating   1000209 non-null  int64
dtypes: int64(3)
memory usage: 22.9 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non

In [4]:
# Check for missing values
print(df1.isnull().sum())
print(df2.isnull().sum())
print(df3.isnull().sum())

MovieID       0
Title         0
Category      0
Unnamed: 3    0
Unnamed: 4    0
dtype: int64
UserID     0
MovieID    0
Rating     0
dtype: int64
UserID        0
Gender        0
Age           0
Occupation    0
dtype: int64


**2.Data Analysis and Mining**

In [5]:
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

**i) Total number of movies released in each year**


In [6]:
import re

df1['YearOfRelease'] = df1['Title'].apply(lambda x: re.search(r'\((\d{4})\)', x).group(1) if re.search(r'\((\d{4})\)', x) else None)
movies_per_year = df1.groupby('YearOfRelease').size().reset_index(name='Count')
print(movies_per_year)


   YearOfRelease  Count
0           1919      2
1           1920      2
2           1921      1
3           1922      2
4           1923      3
..           ...    ...
76          1996    327
77          1997    300
78          1998    323
79          1999    273
80          2000    150

[81 rows x 2 columns]


In [9]:
# Define age groups
bins = [0, 18, 25, 35, 45, 50, 56, 100]
labels = ['0-18', '19-25', '26-35', '36-45', '46-50', '51-56', '57+']
df3['AgeGroup'] = pd.cut(df3['Age'], bins, labels=labels)
print(df3['AgeGroup'])

0        0-18
1       51-56
2       19-25
3       36-45
4       19-25
        ...  
6035    19-25
6036    36-45
6037    51-56
6038    36-45
6039    19-25
Name: AgeGroup, Length: 6040, dtype: category
Categories (7, object): ['0-18' < '19-25' < '26-35' < '36-45' < '46-50' < '51-56' < '57+']


**ii) Movie category having highest ratings in each year**

In [10]:
# Merge datasets
merged_df = df2.merge(df1, on='MovieID').merge(df3, on='UserID')

import streamlit as st    
# Find highest rated category per year
highest_rated_per_year = merged_df.groupby(['YearOfRelease', 'Category'])['Rating'].mean().reset_index()
highest_rated_per_year = highest_rated_per_year.sort_values(['YearOfRelease', 'Rating'], ascending=[True, False]).groupby('YearOfRelease').first().reset_index()
print(highest_rated_per_year)

   YearOfRelease                          Category    Rating
0           1919                            Comedy  3.631579
1           1920                            Comedy  3.666667
2           1921                            Action  3.790323
3           1922                            Horror  3.991597
4           1923                            Comedy  3.444444
..           ...                               ...       ...
76          1996              Crime|Drama|Thriller  4.254676
77          1997  Crime|Film-Noir|Mystery|Thriller  4.219406
78          1998                  Action|Drama|War  4.119213
79          1999       Animation|Children's|Comedy  4.218927
80          2000                      Action|Drama  4.106029

[81 rows x 3 columns]


**iii) Movie category and age group wise likings**

In [11]:
# Find the number of users from each age group who rated each category
age_group_likings = merged_df.groupby(['Category', 'AgeGroup']).size().reset_index(name='Count')


**iv) Cluster models to segregate movie category and age group wise likings**

In [12]:
# Cluster models to segregate movie category and age group wise likings
category_agegroup_counts = merged_df.groupby(['Category', 'AgeGroup']).size().unstack(fill_value=0)
X = category_agegroup_counts.values

# Apply KMeans clustering
kmeans = KMeans(n_clusters=5, random_state=0).fit(X)
category_agegroup_counts['Cluster'] = kmeans.labels_

print(category_agegroup_counts)

AgeGroup                          0-18  19-25  26-35  36-45  46-50  51-56  \
Category                                                                    
 3D (1982)                          44    104     31     12      5      3   
 A Cinderella Story (1998)         132    167     65     28     19      5   
 A Journal of Murder (1995)          0      5      2      0      1      0   
 A Mediaeval Odyssey, The (1988)    13     31     31     18     13      4   
 A New Beginning (1985)             33     57     14      6      4      0   
...                                ...    ...    ...    ...    ...    ...   
Sci-Fi|Thriller                   1077   1873    946    379    292    135   
Sci-Fi|Thriller|War                 20     78     74     40     46     22   
Thriller                          3297   7408   3603   1510   1329    644   
War                                 96    198    228    156    191    122   
Western                            665   1879   1272    626    791    456   

**v) Display year wise count of movies released**

In [13]:
print(movies_per_year)


   YearOfRelease  Count
0           1919      2
1           1920      2
2           1921      1
3           1922      2
4           1923      3
..           ...    ...
76          1996    327
77          1997    300
78          1998    323
79          1999    273
80          2000    150

[81 rows x 2 columns]


**vi) Display year wise, category wise count of movies released**

In [14]:
movies_per_year_category = df1.groupby(['YearOfRelease', 'Category']).size().reset_index(name='Count')
print(movies_per_year_category)

     YearOfRelease                Category  Count
0             1919         Adventure|Drama      1
1             1919                  Comedy      1
2             1920                  Comedy      2
3             1921                  Action      1
4             1922                   Drama      1
...            ...                     ...    ...
1494          2000  Horror|Sci-Fi|Thriller      1
1495          2000        Romance|Thriller      1
1496          2000                  Sci-Fi      1
1497          2000         Sci-Fi|Thriller      1
1498          2000                Thriller      6

[1499 rows x 3 columns]


**Advanced Models**

**vii) Clustering methods to segregate movie category and occupation**

In [15]:
# Clustering methods to segregate movie category and occupation
category_occupation_counts = merged_df.groupby(['Category', 'Occupation']).size().unstack(fill_value=0)
X = category_occupation_counts.values

# Apply KMeans clustering
kmeans = KMeans(n_clusters=5, random_state=0).fit(X)
category_occupation_counts['Cluster'] = kmeans.labels_

print(category_occupation_counts)


Occupation                           0     1    2    3     4    5    6     7  \
Category                                                                       
 3D (1982)                          36    16    9    7    29    6    2    16   
 A Cinderella Story (1998)          56    46   18   16    80    7   18    28   
 A Journal of Murder (1995)          2     0    0    1     0    0    0     2   
 A Mediaeval Odyssey, The (1988)    13     8    8    3     9    5    3    13   
 A New Beginning (1985)             21     7    6    4    18    5    2     6   
...                                ...   ...  ...  ...   ...  ...  ...   ...   
Sci-Fi|Thriller                    617   328  202  122   682  104  146   451   
Sci-Fi|Thriller|War                 36    26   11   13    13    8   10    26   
Thriller                          2411  1444  836  632  2143  394  719  2058   
War                                106    98   28   23    81   13   39   143   
Western                            751  

**viii) Refine the model by including age group**

In [16]:
# Refine the model by including age group
category_agegroup_occupation_counts = merged_df.groupby(['Category', 'AgeGroup', 'Occupation']).size().unstack(fill_value=0).fillna(0)
X = category_agegroup_occupation_counts.values

# Apply KMeans clustering
kmeans = KMeans(n_clusters=5, random_state=0).fit(X)
category_agegroup_occupation_counts['Cluster'] = kmeans.labels_

print(category_agegroup_occupation_counts)

Occupation             0    1   2   3   4   5   6    7  8   9  ...  12  13  \
Category   AgeGroup                                            ...           
 3D (1982) 0-18        5    3   0   1  18   1   0    1  0   0  ...   4   0   
           19-25      20   10   8   4  10   3   0    8  0   1  ...   9   0   
           26-35       5    3   1   1   1   2   1    4  0   2  ...   0   0   
           36-45       4    0   0   0   0   0   0    1  0   0  ...   0   0   
           46-50       2    0   0   1   0   0   1    0  0   0  ...   0   0   
...                  ...  ...  ..  ..  ..  ..  ..  ... ..  ..  ...  ..  ..   
Western    26-35     189  113  48  43  24  39  34  233  0  20  ...  56  18   
           36-45      53   67  41  18   0  30  32  105  0   4  ...  23   8   
           46-50     119   65  34  35   2   2  56  170  5   8  ...  21  22   
           51-56      36   48   3  39   0   0   9   55  1   3  ...  11  86   
           57+         0    0   0   0   0   0   0    0  0   0  .

**ix) Predictive model based on category, age group, and occupation**

In [19]:
# Predictive model based on category, age group, and occupation
category_agegroup_occupation_counts = merged_df.groupby(['Category', 'AgeGroup', 'Occupation']).size().reset_index(name='Count')
X_pred = category_agegroup_occupation_counts[['Category', 'AgeGroup', 'Occupation']]
y_pred = category_agegroup_occupation_counts['Count']

X_pred = pd.get_dummies(X_pred)

X_train, X_test, y_train, y_test = train_test_split(X_pred, y_pred, test_size=0.2, random_state=0)

model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)

y_pred_model = model.predict(X_test)


**Develop a Simple User Interface**

In [21]:
import streamlit as st
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import re
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

# Define file paths
csv_file1_path = 'Movies.csv'
csv_file2_path = 'Ratings.csv'
csv_file3_path = 'Users.csv'

# Detect file encodings
encoding1 = detect_encoding(csv_file1_path)
encoding2 = detect_encoding(csv_file2_path)
encoding3 = detect_encoding(csv_file3_path)

# Load data with detected encodings
df1 = pd.read_csv(csv_file1_path, encoding=encoding1)
df2 = pd.read_csv(csv_file2_path, encoding=encoding2)
df3 = pd.read_csv(csv_file3_path, encoding=encoding3)

# Clean data
df1.drop_duplicates(inplace=True)
df2.drop_duplicates(inplace=True)
df3.drop_duplicates(inplace=True)

df1.fillna('', inplace=True)
df2.fillna(0, inplace=True)
df3.fillna('', inplace=True)

# Extract Year of Release from Title
df1['YearOfRelease'] = df1['Title'].apply(lambda x: re.search(r'\((\d{4})\)', x).group(1) if re.search(r'\((\d{4})\)', x) else None)
movies_per_year = df1.groupby('YearOfRelease').size().reset_index(name='Count')

# Define age groups
bins = [0, 18, 25, 35, 45, 50, 56, 100]
labels = ['0-18', '19-25', '26-35', '36-45', '46-50', '51-56', '57+']
df3['AgeGroup'] = pd.cut(df3['Age'], bins, labels=labels)

# Merge datasets
merged_df = df2.merge(df1, on='MovieID').merge(df3, on='UserID')

# Find highest rated category per year
highest_rated_per_year = merged_df.groupby(['YearOfRelease', 'Category'])['Rating'].mean().reset_index()
highest_rated_per_year = highest_rated_per_year.sort_values(['YearOfRelease', 'Rating'], ascending=[True, False]).groupby('YearOfRelease').first().reset_index()

# Find the number of users from each age group who rated each category
age_group_likings = merged_df.groupby(['Category', 'AgeGroup']).size().reset_index(name='Count')

# Cluster models to segregate movie category and age group wise likings
category_agegroup_counts = merged_df.groupby(['Category', 'AgeGroup']).size().unstack(fill_value=0)
X_agegroup = category_agegroup_counts.values

# Apply KMeans clustering
kmeans_agegroup = KMeans(n_clusters=5, random_state=0).fit(X_agegroup)
category_agegroup_counts['Cluster'] = kmeans_agegroup.labels_

# Clustering methods to segregate movie category and occupation
category_occupation_counts = merged_df.groupby(['Category', 'Occupation']).size().unstack(fill_value=0)
X_occupation = category_occupation_counts.values

# Apply KMeans clustering
kmeans_occupation = KMeans(n_clusters=5, random_state=0).fit(X_occupation)
category_occupation_counts['Cluster'] = kmeans_occupation.labels_

# Refine the model by including age group
category_agegroup_occupation_counts = merged_df.groupby(['Category', 'AgeGroup', 'Occupation']).size().unstack(fill_value=0).fillna(0)
X_combined = category_agegroup_occupation_counts.values

# Apply KMeans clustering
kmeans_combined = KMeans(n_clusters=5, random_state=0).fit(X_combined)
category_agegroup_occupation_counts['Cluster'] = kmeans_combined.labels_

# Predictive model based on category, age group, and occupation
category_agegroup_occupation_counts = merged_df.groupby(['Category', 'AgeGroup', 'Occupation']).size().reset_index(name='Count')
X_pred = category_agegroup_occupation_counts[['Category', 'AgeGroup', 'Occupation']]
y_pred = category_agegroup_occupation_counts['Count']

X_pred = pd.get_dummies(X_pred)

# Use a smaller subset of the data for training
X_train, X_test, y_train, y_test = train_test_split(X_pred, y_pred, test_size=0.9, random_state=0)

model = RandomForestClassifier(n_estimators=50, random_state=0)
model.fit(X_train, y_train)

y_pred_model = model.predict(X_test)

# Streamlit UI
st.title("Movie Analysis and Prediction System")

# Query 1: Total number of movies released in each year
st.header("Total number of movies released in each year")
st.dataframe(movies_per_year)

# Query 2: Movie category having highest ratings in each year
st.header("Movie category having highest ratings in each year")
st.dataframe(highest_rated_per_year)

# Query 3: Movie category and age group wise likings
st.header("Movie category and age group wise likings")
st.dataframe(age_group_likings)

# Query 4: Cluster models to segregate movie category and age group wise likings
st.header("Cluster models to segregate movie category and age group wise likings")
st.dataframe(category_agegroup_counts)

# Query 5: Year wise count of movies released
st.header("Year wise count of movies released")
st.dataframe(movies_per_year)

# Query 6: Year wise, category wise count of movies released
st.header("Year wise, category wise count of movies released")
movies_per_year_category = df1.groupby(['YearOfRelease', 'Category']).size().reset_index(name='Count')
st.dataframe(movies_per_year_category)

# Query 7: Clustering methods to segregate movie category and occupation
st.header("Clustering methods to segregate movie category and occupation")
st.dataframe(category_occupation_counts)

# Query 8: Refine the model by including age group
st.header("Refined model by including age group and occupation")
st.dataframe(category_agegroup_occupation_counts)

# Query 9: Predictive model based on category, age group, and occupation
st.header("Predictive model based on category, age group, and occupation")
st.write('Model Accuracy:', accuracy_score(y_test, y_pred_model))

# User inputs for prediction
st.header("Predict movie likings based on user inputs")
category_input = st.selectbox("Select Category", df1['Category'].unique())
agegroup_input = st.selectbox("Select Age Group", labels)
occupation_input = st.selectbox("Select Occupation", range(21))

if st.button("Predict"):
    user_input = pd.DataFrame({
        'Category': [category_input],
        'AgeGroup': [agegroup_input],
        'Occupation': [occupation_input]
    })
    user_input = pd.get_dummies(user_input)
    
    # Ensure all columns in training data are in user input
    for col in X_train.columns:
        if col not in user_input.columns:
            user_input[col] = 0
    
    # Reorder columns to match the training set
    user_input = user_input[X_train.columns]
    
    prediction = model.predict(user_input)
    st.write(f'Predicted number of users: {prediction[0]}')


C:\Users\tauqu\AppData\Roaming\Python\Python310\site-packages\streamlit\type_util.py:1084: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = pa.Table.from_pandas(df)
